# **2일차 팀 프로젝트: 보안 문서 기반 RAG 시스템 구축**

## 프로젝트 목표
1. 보안 표준/가이드 PDF 6종(`datasets/보안 pdf모음`)을 Qdrant Cloud에 저장
2. Parent Document Retriever 패턴 적용
3. 검색 테스트 및 보안 도메인 RAG 시스템 구현

## 구현 단계
- 환경 설정 확인
- 보안 PDF 문서 로딩 (CVSS, CNA Rules, NIST SP 800-126r4/800-216, CISA 플레이북 등)
- Child Chunk 생성 및 Qdrant Cloud 저장
- Parent Document 저장
- 검색 테스트
- RAG 시스템 구현 및 테스트

## 0. 환경 변수 설정

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# API Key 확인
if os.environ.get("OPENAI_API_KEY"):
    print("✓ OpenAI API Key가 설정되었습니다.")
else:
    print("✗ OpenAI API Key가 없습니다.")

# Qdrant Cloud 설정 확인
if os.environ.get("QDRANT_URL") and os.environ.get("QDRANT_API_KEY"):
    print("✓ Qdrant Cloud 설정이 완료되었습니다.")
    print(f"  URL: {os.environ.get('QDRANT_URL')}")
else:
    print("✗ Qdrant Cloud 설정이 필요합니다.")
    print("  .env 파일에 QDRANT_URL과 QDRANT_API_KEY를 추가하세요.")

## 1. PDF 문서 로딩

**선정 문서: `../datasets/보안 pdf모음` 폴더의 보안 표준/가이드 PDF 6종**

- `CNA_Rules_v4.1.0.pdf` — CVE Numbering Authority(CNA) 운영 규칙
- `cvss-v40-specification.pdf` — CVSS v4.0 취약점 심각도 평가 명세
- `Federal_Government_Cybersecurity_Incident_and_Vulnerability_Response_Playbooks_508C.pdf` — 미 연방정부 사이버 사고/취약점 대응 플레이북 (CISA)
- `Key-Details-Phrasing.pdf` — CVE 설명 작성(Key Details Phrasing) 가이드
- `NIST.SP.800-126r4.pdf` — SCAP 1.3 명세 (NIST)
- `NIST.SP.800-216.pdf` — 취약점 공개(Vulnerability Disclosure) 연방 가이드라인 (NIST)

In [ ]:
from langchain_core.documents import Document
from pathlib import Path
import fitz

# 보안 PDF 모음 폴더의 모든 PDF 로딩
pdf_dir = Path("../datasets/보안 pdf모음")
pdf_files = sorted(pdf_dir.glob("*.pdf"))

# [개선 4] 메타데이터 활용: 문서별 카테고리 부여 (검색 시 필터링에 사용)
CATEGORY_MAP = {
    "CNA_Rules_v4.1.0.pdf": "CVE 관리",
    "cvss-v40-specification.pdf": "취약점 평가",
    "Federal_Government_Cybersecurity_Incident_and_Vulnerability_Response_Playbooks_508C.pdf": "사고 대응",
    "Key-Details-Phrasing.pdf": "CVE 작성 가이드",
    "NIST.SP.800-126r4.pdf": "SCAP 표준",
    "NIST.SP.800-216.pdf": "취약점 공개",
}

print(f"로딩할 PDF 파일 ({len(pdf_files)}개):")
for f in pdf_files:
    print(f"  - {f.name}  [{CATEGORY_MAP.get(f.name, '기타')}]")

docs = []

for pdf_path in pdf_files:
    doc = fitz.open(pdf_path)
    file_name = pdf_path.name

    # 페이지 단위로 Document 생성 (Parent Document)
    for page_num in range(len(doc)):
        page = doc[page_num]
        text = page.get_text("text", sort=True)

        # 빈 페이지는 스킵
        if len(text.strip()) < 10:
            continue

        docs.append(
            Document(
                page_content=text,
                metadata={
                    "source": file_name,
                    "category": CATEGORY_MAP.get(file_name, "기타"),
                    "page": page_num + 1,
                    # 파일이 여러 개이므로 파일명을 포함해 고유한 parent_id 생성
                    "parent_id": f"{pdf_path.stem}_page_{page_num + 1}"
                }
            )
        )

    doc.close()

print(f"\n총 {len(docs)}개의 페이지(Parent Document) 로드 완료")
print(f"\n첫 번째 페이지 길이: {len(docs[0].page_content)}자")
print(f"평균 페이지 길이: {sum(len(d.page_content) for d in docs) / len(docs):.0f}자")

# 첫 페이지 내용 미리보기
print(f"\n첫 페이지 내용 미리보기:")
print(docs[0].page_content[:300] + "...")

## 2. Child Chunk 생성

**TODO: 청킹 전략을 조정해보세요 (선택사항)**
- chunk_size: 각 청크의 크기 (기본 400자)
- chunk_overlap: 청크 간 겹치는 부분 (기본 50자)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# [개선 1] 청킹 전략 최적화
# - chunk_size 400 → 600: 표/목록이 많은 기술 문서에서 문맥이 잘리는 것을 완화
# - chunk_overlap 50 → 100: 청크 경계에서 정의/문장이 끊기는 것을 방지
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100
)

# Parent를 Child chunk로 분할
child_docs = []

for parent_doc in docs:
    chunks = child_splitter.split_text(parent_doc.page_content)

    for chunk in chunks:
        child_docs.append(
            Document(
                page_content=chunk,
                metadata={
                    "parent_id": parent_doc.metadata["parent_id"],
                    "page": parent_doc.metadata["page"],
                    "source": parent_doc.metadata["source"],
                    "category": parent_doc.metadata["category"]
                }
            )
        )

print(f"\n생성된 통계:")
print(f"  - Parent 문서 수: {len(docs)}")
print(f"  - Child chunk 수: {len(child_docs)}")
print(f"  - 평균 chunk/page: {len(child_docs) / len(docs):.1f}")

# Child chunk 샘플 확인
print(f"\nChild chunk 샘플 (첫 3개):")
for i in range(min(3, len(child_docs))):
    print(f"\nChunk {i + 1}:")
    print(f"  Parent ID: {child_docs[i].metadata['parent_id']}")
    print(f"  Category: {child_docs[i].metadata['category']}")
    print(f"  Page: {child_docs[i].metadata['page']}")
    print(f"  Length: {len(child_docs[i].page_content)}자")
    print(f"  Content: {child_docs[i].page_content[:100]}...")

## 3. Qdrant Cloud에 Child Chunk 저장

**TODO: 컬렉션 이름을 팀 프로젝트에 맞게 변경하세요**

In [ ]:
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from uuid import uuid4

# Qdrant Cloud 클라이언트 생성
client = QdrantClient(
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY")
)

print("Qdrant Cloud에 연결되었습니다.")
print(f"  URL: {os.getenv('QDRANT_URL')}")

In [ ]:
# 임베딩 함수
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

# 보안 문서 컬렉션
collection_name = "security_docs"

# 컬렉션이 이미 있을 때 삭제 후 새로 만들지 여부
# (청킹 전략을 바꾼 경우에만 True로 두고 다시 업로드)
RECREATE_COLLECTION = False

# 컬렉션 존재 여부 확인
collections = client.get_collections().collections
existing_collection = any(col.name == collection_name for col in collections)

need_upload = False

if existing_collection and RECREATE_COLLECTION:
    print(f"기존 컬렉션 '{collection_name}' 삭제 중...")
    client.delete_collection(collection_name=collection_name)
    existing_collection = False
    print("컬렉션이 삭제되었습니다.")

if not existing_collection:
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=3072, distance=Distance.COSINE)
    )
    need_upload = True
    print(f"컬렉션 '{collection_name}' 생성 완료")
else:
    print(f"기존 컬렉션 '{collection_name}'을 사용합니다.")

# [개선 4] 메타데이터 필터링용 payload 인덱스 생성
# Qdrant Cloud는 인덱스 없는 필드로 필터링할 수 없음 (이미 있으면 그대로 유지됨)
for field in ["metadata.category", "metadata.source"]:
    client.create_payload_index(
        collection_name=collection_name,
        field_name=field,
        field_schema="keyword"
    )
print("payload 인덱스 확인 완료: metadata.category, metadata.source")

# 벡터스토어 생성
vectorstore = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embeddings
)

# Child chunk 추가 (새로 만든 컬렉션일 때만 업로드)
if need_upload:
    uuids = [str(uuid4()) for _ in range(len(child_docs))]
    vectorstore.add_documents(documents=child_docs, ids=uuids)
    print(f"\n{len(child_docs)}개의 Child chunk가 Qdrant Cloud에 추가되었습니다.")
else:
    print(f"\n기존 데이터를 사용합니다. (현재 {client.count(collection_name).count}개 포인트)")

## 4. Parent Document 저장 (Docstore)

In [ ]:
# Parent 문서를 dict에 저장 (parent_id를 키로 사용)
parent_docstore = {}

for parent_doc in docs:
    parent_id = parent_doc.metadata["parent_id"]
    parent_docstore[parent_id] = parent_doc

print(f"Docstore에 {len(parent_docstore)}개의 Parent 문서 저장 완료")
print(f"\nDocstore 키 예시: {list(parent_docstore.keys())[:5]}")

## 5. Parent Document Retriever 구현

In [ ]:
from typing import List, Optional
from qdrant_client.http import models as qmodels

class ParentDocumentRetriever:
    """
    Parent Document Retriever 직접 구현

    원리:
    1. vectorstore에서 child chunk 검색
    2. child chunk의 parent_id 추출
    3. docstore에서 parent_id로 parent 문서 반환

    [개선 2] k 값 2 → 4: 여러 문서에 걸친 질문에 더 많은 근거 페이지 제공
    [개선 4] category/source 메타데이터 필터링 지원
    """

    def __init__(self, vectorstore, parent_docstore, k: int = 4):
        self.vectorstore = vectorstore
        self.parent_docstore = parent_docstore
        self.k = k

    @staticmethod
    def _build_filter(category: Optional[str] = None, source: Optional[str] = None):
        """Qdrant 메타데이터 필터 생성 (category, source)"""
        conditions = []
        if category:
            conditions.append(qmodels.FieldCondition(
                key="metadata.category",
                match=qmodels.MatchValue(value=category)
            ))
        if source:
            conditions.append(qmodels.FieldCondition(
                key="metadata.source",
                match=qmodels.MatchValue(value=source)
            ))
        return qmodels.Filter(must=conditions) if conditions else None

    def invoke(self, query: str, category: Optional[str] = None,
               source: Optional[str] = None) -> List[Document]:
        # 1. Vectorstore에서 child chunk 검색 (parent 중복을 고려해 넉넉히 검색)
        qdrant_filter = self._build_filter(category, source)
        child_results = self.vectorstore.similarity_search(
            query, k=self.k * 3, filter=qdrant_filter
        )

        # 2. Child chunk에서 parent_id 추출 (중복 제거)
        parent_ids = []
        for doc in child_results:
            parent_id = doc.metadata.get("parent_id")
            if parent_id and parent_id not in parent_ids:
                parent_ids.append(parent_id)
                if len(parent_ids) >= self.k:
                    break

        # 3. Docstore에서 parent 문서 가져오기
        parent_docs = []
        for parent_id in parent_ids:
            if parent_id in self.parent_docstore:
                parent_docs.append(self.parent_docstore[parent_id])

        return parent_docs

    def get_child_chunks(self, query: str, k: int = 3,
                         category: Optional[str] = None,
                         source: Optional[str] = None) -> List[Document]:
        """비교용: Child chunk 직접 반환"""
        qdrant_filter = self._build_filter(category, source)
        return self.vectorstore.similarity_search(query, k=k, filter=qdrant_filter)

# Retriever 생성
parent_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    parent_docstore=parent_docstore,
    k=4
)

print("✓ Parent Document Retriever 생성 완료 (k=4, 메타데이터 필터 지원)")

## 5-1. 하이브리드 검색 (BM25 + 벡터, RRF 융합)

**[개선 5]** 키워드 검색(BM25)과 벡터 검색을 결합합니다.
- **BM25**: `CVE`, `SCAP`, `CVSS-BT` 같은 정확한 용어·식별자 매칭에 강함
- **벡터 검색**: 의미가 비슷한 표현(한국어 질문 ↔ 영어 문서)에 강함
- **RRF(Reciprocal Rank Fusion)**: 두 검색 결과의 순위를 `1/(60+rank)` 점수로 합산해 융합

In [ ]:
from rank_bm25 import BM25Okapi
import re

def tokenize(text: str) -> List[str]:
    """영문/숫자/한글 토큰 추출 (소문자화)"""
    return re.findall(r"[a-z0-9]+|[가-힣]+", text.lower())

# Child chunk 전체로 BM25 인덱스 구축 (로컬, 무료)
bm25_index = BM25Okapi([tokenize(d.page_content) for d in child_docs])

class HybridParentRetriever:
    """
    [개선 5] 하이브리드 검색: BM25(키워드) + 벡터 검색을 RRF로 결합

    원리:
    1. 벡터 검색과 BM25 검색을 각각 수행해 parent_id 순위를 얻음
    2. Reciprocal Rank Fusion(RRF)으로 두 순위를 융합: score = Σ 1/(rrf_k + rank)
    3. 융합 점수 상위 k개의 parent 문서 반환

    CVE-2024-1234, "SCAP", "CVSS-BT" 같은 정확한 용어/식별자는 BM25가,
    의미 기반 질문은 벡터 검색이 강함 → 둘을 합쳐 상호 보완
    """

    def __init__(self, vectorstore, parent_docstore, child_docs, bm25_index,
                 k: int = 4, candidate_k: int = 12, rrf_k: int = 60):
        self.vectorstore = vectorstore
        self.parent_docstore = parent_docstore
        self.child_docs = child_docs
        self.bm25_index = bm25_index
        self.k = k                      # 최종 반환할 parent 수
        self.candidate_k = candidate_k  # 각 검색기가 뽑을 child 후보 수
        self.rrf_k = rrf_k              # RRF 완충 상수

    def _vector_parent_ranking(self, query, category=None):
        """벡터 검색 → parent_id 순위 리스트"""
        qdrant_filter = ParentDocumentRetriever._build_filter(category)
        results = self.vectorstore.similarity_search(
            query, k=self.candidate_k, filter=qdrant_filter
        )
        ranking = []
        for doc in results:
            pid = doc.metadata.get("parent_id")
            if pid and pid not in ranking:
                ranking.append(pid)
        return ranking

    def _bm25_parent_ranking(self, query, category=None):
        """BM25 키워드 검색 → parent_id 순위 리스트"""
        scores = self.bm25_index.get_scores(tokenize(query))
        order = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
        ranking = []
        for i in order:
            if scores[i] <= 0:
                break
            doc = self.child_docs[i]
            if category and doc.metadata.get("category") != category:
                continue
            pid = doc.metadata.get("parent_id")
            if pid and pid not in ranking:
                ranking.append(pid)
            if len(ranking) >= self.candidate_k:
                break
        return ranking

    def invoke(self, query: str, category: Optional[str] = None) -> List[Document]:
        vector_rank = self._vector_parent_ranking(query, category)
        bm25_rank = self._bm25_parent_ranking(query, category)

        # RRF 융합
        rrf_scores = {}
        for ranking in (vector_rank, bm25_rank):
            for rank, pid in enumerate(ranking, start=1):
                rrf_scores[pid] = rrf_scores.get(pid, 0) + 1 / (self.rrf_k + rank)

        top_ids = sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:self.k]
        return [self.parent_docstore[pid] for pid in top_ids
                if pid in self.parent_docstore]

# 하이브리드 Retriever 생성
hybrid_retriever = HybridParentRetriever(
    vectorstore=vectorstore,
    parent_docstore=parent_docstore,
    child_docs=child_docs,
    bm25_index=bm25_index,
    k=4
)

print("✓ Hybrid Parent Retriever 생성 완료 (BM25 + 벡터, RRF 융합)")

## 6. 검색 테스트

**TODO: 팀 문서에 맞는 질문으로 변경하여 검색 테스트를 진행하세요**

In [ ]:
# 보안 문서에 맞는 검색 질문
query = "CVSS v4.0에서 취약점 심각도 등급(Critical, High, Medium, Low)은 어떤 점수 범위로 나뉘나요?"

def print_parents(results):
    for i, result in enumerate(results, start=1):
        print(f"\n  [{i}] {result.metadata.get('source', '?')} "
              f"(p.{result.metadata.get('page', '?')}, "
              f"카테고리: {result.metadata.get('category', '?')})")
        print(f"      미리보기: {result.page_content[:150].strip()}...")

print(f"검색 쿼리: {query}\n")
print("="*80)

# [1] Child chunk 검색
print("\n[1] Child Chunk 검색 결과 (벡터)")
print("-"*80)
child_results = parent_retriever.get_child_chunks(query, k=2)
for i, result in enumerate(child_results, start=1):
    print(f"\n  Chunk {i}: {result.metadata.get('source', '?')} "
          f"(p.{result.metadata.get('page', '?')})")
    print(f"  내용: {result.page_content[:300]}")

# [2] Parent 검색 (벡터만)
print("\n" + "="*80)
print("\n[2] Parent Document 검색 결과 (벡터만)")
print("-"*80)
print_parents(parent_retriever.invoke(query))

# [3] Parent 검색 (하이브리드: BM25 + 벡터)
print("\n" + "="*80)
print("\n[3] Parent Document 검색 결과 (하이브리드: BM25 + 벡터, RRF)")
print("-"*80)
print_parents(hybrid_retriever.invoke(query))

# [4] 메타데이터 필터 검색: '취약점 평가' 카테고리(CVSS 문서)로 한정
print("\n" + "="*80)
print("\n[4] 카테고리 필터 적용 검색 (category='취약점 평가')")
print("-"*80)
print_parents(hybrid_retriever.invoke(query, category="취약점 평가"))

## 7. RAG 시스템 구현

**TODO: 시스템 프롬프트를 팀 문서에 맞게 수정하세요**

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from IPython.display import Markdown, display

llm = init_chat_model("gpt-5.4-mini")

# [개선 3] 프롬프트 개선: 답변 형식을 구체적으로 지정
template = """
당신은 사이버 보안 표준 및 취약점 관리 전문가입니다.
CVE/CNA 규칙, CVSS 취약점 평가, NIST 보안 표준(SCAP, 취약점 공개 가이드라인),
사이버 사고 대응 플레이북에 대한 깊은 지식을 갖추고 있습니다.

주어진 [참고 정보]만을 근거로 답변하세요.
참고 문서가 영어라도 한국어로 알기 쉽게 설명하되, 핵심 용어는 원문(영어)을 병기하세요.
참고 정보에 없는 내용은 추측하지 말고 "제공된 문서에서 찾을 수 없습니다"라고 답하세요.

반드시 아래 형식으로 답변하세요:

### 핵심 답변
(질문에 대한 직접적인 답을 2~3문장으로 요약)

### 상세 설명
(근거가 되는 내용을 목록이나 표로 구조화하여 설명.
 항목별 기준·수치·단계가 있으면 표로 정리)

### 출처
(참고한 문서명과 페이지 번호를 목록으로: - 문서명, p.페이지)

[참고 정보]
{context}

[질문]
{question}
"""

prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template=template
)

def rag_with_parent_retriever(question: str, category: str = None,
                              use_hybrid: bool = True) -> str:
    """
    Parent Document Retriever를 사용한 RAG
    - use_hybrid=True: [개선 5] BM25 + 벡터 하이브리드 검색 사용
    - category: [개선 4] 특정 카테고리 문서로 검색 범위 한정
    """
    # 1. 문서 검색
    retriever = hybrid_retriever if use_hybrid else parent_retriever
    retrieved_docs = retriever.invoke(question, category=category)

    # 2. 컨텍스트 구성
    context_parts = []
    for doc in retrieved_docs:
        source = doc.metadata.get("source", "?")
        page_num = doc.metadata.get("page", "?")
        cat = doc.metadata.get("category", "?")
        context_parts.append(
            f"[출처: {source}, 페이지: {page_num}, 카테고리: {cat}]\n{doc.page_content}"
        )

    context = "\n\n---\n\n".join(context_parts)

    # 3. 프롬프트 생성
    formatted_prompt = prompt_template.format(
        context=context,
        question=question
    )

    # 4. LLM 호출
    response = llm.invoke(formatted_prompt)
    return response.content

print("✓ RAG 시스템 준비 완료 (하이브리드 검색 + 구조화된 답변 형식)")

## 8. RAG 시스템 테스트

**TODO: 팀 문서에 맞는 다양한 질문으로 RAG 시스템을 테스트하세요**

In [ ]:
# 보안 문서에 맞는 테스트 질문
questions = [
    "CVSS v4.0의 Base Metric에는 어떤 항목들이 있고 각각 무엇을 평가하나요?",
    "CNA가 CVE ID를 할당할 수 있는 취약점의 기준은 무엇인가요?",
    "연방정부 사이버 사고 대응 플레이북에서 정의하는 사고 대응(Incident Response) 단계는 무엇인가요?"
]

for q in questions:
    print(f"\n{'='*80}")
    print(f"질문: {q}")
    print(f"{'='*80}\n")

    answer = rag_with_parent_retriever(q)
    display(Markdown(answer))

## 프로젝트 점검 체크리스트

**완료한 항목을 확인하세요:**

- [x] PDF 문서 선정 및 로딩 완료 (보안 PDF 6종)
- [x] Child Chunk 생성 완료
- [x] Qdrant Cloud에 데이터 저장 완료
- [x] Parent Document Retriever 구현 완료
- [x] 검색 테스트 완료 (Child vs Parent vs 하이브리드 비교)
- [x] RAG 시스템 구현 완료
- [x] 최소 3개 이상의 질문으로 테스트 완료
- [x] 시스템 프롬프트 도메인에 맞게 수정 완료

---

## 추가 개선 아이디어 (적용 완료)

1. ~~**청킹 전략 최적화**~~ ✅ chunk_size 400→600, chunk_overlap 50→100 (섹션 2)
2. ~~**검색 개수 조정**~~ ✅ retriever k 값 2→4 (섹션 5)
3. ~~**프롬프트 개선**~~ ✅ 핵심 답변/상세 설명/출처 3단 형식 지정 (섹션 7)
4. ~~**메타데이터 활용**~~ ✅ 문서별 category 메타데이터 + Qdrant 필터 검색 (섹션 1, 5, 6)
5. ~~**하이브리드 검색**~~ ✅ BM25 + 벡터 검색 RRF 융합 (섹션 5-1)

### 다음 단계 아이디어
- 리랭커(reranker) 모델로 검색 결과 재정렬
- 질문에서 카테고리를 자동 추론해 필터 적용 (query routing)
- 대화 이력을 반영한 멀티턴 RAG